# Inspect test-mode samples
Run `bash run_labels.sh <DATASET> test` and `bash run_inputs.sh <DATASET> test` first. This notebook shows the outputs in `Outputs/<DATASET>/Sample/labels/` and `Outputs/<DATASET>/Sample/inputs/` so we can check their quality before the full runs.

In [ ]:
import sys
import textwrap
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
sys.path.append("../Config")
from config import SCALE_THRESHOLD, ori_dir

DATASET = "FHM"  # or "MAMI"
labels_dir = f"Outputs/{DATASET}/Sample/labels/"
inputs_dir = f"Outputs/{DATASET}/Sample/inputs/"

### Labels: teacher scale scores

In [ ]:
df = pd.read_json(labels_dir + "ex_train_scale.jsonl", lines=True, dtype={"id": str})
for prompt in ["simple_scale", "category_scale"]:
    agree = (df.label == (df[prompt] >= SCALE_THRESHOLD).astype(int)) & df[prompt].notna()
    print(f"{prompt}: agreement with binary label {agree.sum()}/{len(df)} ({100*agree.mean():.1f}%), unparsable {df[prompt].isna().sum()}")
    display(pd.crosstab(df[prompt], df.label, margins=True))

In [ ]:
n_cols = 5
n_rows = (len(df) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 4.6 * n_rows))
for ax in axes.flat:
    ax.axis("off")
for ax, (_, r) in zip(axes.flat, df.iterrows()):
    ax.imshow(Image.open(ori_dir(DATASET) + r.img).convert("RGB"))
    ax.set_title(f"label {r.label} | simple {r.simple_scale} | category {r.category_scale}\n" + textwrap.shorten(r.text, 60), fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# raw teacher responses
pd.set_option("display.max_colwidth", 200)
df[["id", "text", "label", "simple_scale_text", "category_scale_text"]]

### Inputs: original hateful meme vs regenerated non-hateful meme

In [ ]:
ex = pd.read_json(inputs_dir + "ex_train.jsonl", lines=True, dtype={"id": str})
print(f"{len(ex)} regenerated memes")

for _, r in ex.iterrows():
    fig, axes = plt.subplots(1, 2, figsize=(11, 5.5))
    axes[0].imshow(Image.open(ori_dir(DATASET) + r.orig_img).convert("RGB"))
    axes[0].set_title("Original (hateful)\n" + textwrap.fill(r.orig_text, 60), fontsize=9)
    axes[1].imshow(Image.open(inputs_dir + r.img).convert("RGB"))
    axes[1].set_title("Regenerated (non-hateful)\n" + textwrap.fill(r.text, 60), fontsize=9)
    for ax in axes:
        ax.axis("off")
    fig.suptitle(f"id {r.id} | background: " + textwrap.shorten(r.desc, 160), fontsize=8)
    plt.tight_layout()
    plt.show()